### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="cirrhosis_patient_survival_prediction",
    dataset_year="1984",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5R02G",
    download_description="""
We get the UCI data.

wget https://archive.ics.uci.edu/static/public/878/cirrhosis+patient+survival+prediction+dataset-1.zip && unzip cirrhosis+patient+survival+prediction+dataset-1.zip && rm cirrhosis+patient+survival+prediction+dataset-1.zip && mkdir -p local-data-warehouse/cirrhosis_patient_survival_prediction && mv cirrhosis.csv local-data-warehouse/cirrhosis_patient_survival_prediction/
""",
    # References
    academic_reference_bibtex="""@article{dickson1989prognosis,
  title={Prognosis in primary biliary cirrhosis: model for decision making},
  author={Dickson, E Rolland and Grambsch, Patricia M and Fleming, Thomas R and Fisher, Lloyd D and Langworthy, Alice},
  journal={Hepatology},
  volume={10},
  number={1},
  pages={1--7},
  year={1989},
  publisher={Wiley Online Library}
}
""",
    academic_reference_bibtex_key="dickson1989prognosis",
    license="IID",
    data_tags=["IID"],
    curation_comments="""
We start with the UCI data.

We note that this is a survival prediction task. But the way we split the data and score the task, it is not treated correctly as a survival prediction task. At the same time, it is randomized control data based on the Drug column. We treat it as a regression task to predict the time-to-death for non-censored patients, including the drug column into the modelling.

- We drop ID and Status column as they have no relevance after filtering to only non-censored patients.
- We log transform N_days and rename it to the target "log_days_to_death".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="log_days_to_death",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "cirrhosis.csv")
print("Loaded data shape:", df.shape)

df = df[df["Status"] == "D"]
df["log_days_to_death"] = np.log(df["N_Days"])
df = df.drop(columns=["ID", "Status", "N_Days"])

as_cat_type = ["Drug", "Ascites", "Hepatomegaly", "Spiders", "Edema", "Stage", "Sex"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits


split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()